# Benchmarks Base Notebook

This notebook evaluates a Hugging Face model on lm-eval tasks, measures throughput and memory, and computes energy efficiency (Joules/token).

In [19]:
%pip install -q \
      "torch" \
      "transformers" \
      "accelerate" \
      "lm_eval==0.4.9.1" \
      "sentencepiece==0.2.1" \
      "langdetect" \
      "datasets" \
      "codecarbon" \
      "pandas"

In [20]:
import os
import json

import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

In [21]:
from pathlib import Path
import sys

# Find a utils.py near the current working directory (works in VSCode and Colab)
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
utils_path = None
for base in candidates:
    candidate = base / "utils.py"
    if candidate.exists():
        utils_path = candidate
        if str(base) not in sys.path:
            sys.path.insert(0, str(base))
        break

print(f"Working directory: {cwd}")
print(f"Resolved utils.py: {utils_path}")

Working directory: /content
Resolved utils.py: /content/utils.py


In [22]:
import sys
import types

# Some environments have an optipfair/transformers version mismatch.
# The benchmark functions used in this notebook do not require optipfair directly.
sys.modules.setdefault("optipfair", types.ModuleType("optipfair"))

from utils import model_evaluation, measure_memory_allocation, measure_energy_consumption

print("Imported utils functions successfully.")

Imported utils functions successfully.


In [23]:
# -----------------------------
# Main configuration variables
# -----------------------------
MODEL_NAME = "meta-llama/Llama-3.2-1B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
MAX_NEW_TOKENS = 64

BENCHMARK_TASKS = [
    "arc_easy",
    "winogrande",
    "hellaswag",
    "lambada_openai",
    "piqa",
]

def get_env_int(name: str, default: int) -> int:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    try:
        return int(value)
    except ValueError:
        print(f"Warning: invalid integer for {name}={value!r}. Using default={default}.")
        return default

BENCHMARK_LIMIT = get_env_int("BENCHMARK_LIMIT", 400)

print("Configuration:")
print(f"- MODEL_NAME: {MODEL_NAME}")
print(f"- DEVICE: {DEVICE}")
print(f"- BATCH_SIZE: {BATCH_SIZE}")
print(f"- MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
print(f"- BENCHMARK_LIMIT (env BENCHMARK_LIMIT): {BENCHMARK_LIMIT}")
print(f"- BENCHMARK_TASKS: {BENCHMARK_TASKS}")

Configuration:
- MODEL_NAME: meta-llama/Llama-3.2-1B
- DEVICE: cuda
- BATCH_SIZE: 4
- MAX_NEW_TOKENS: 64
- BENCHMARK_LIMIT (env BENCHMARK_LIMIT): 400
- BENCHMARK_TASKS: ['arc_easy', 'winogrande', 'hellaswag', 'lambada_openai', 'piqa']


In [24]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print("Model loaded successfully:")
print(f"- Name: {MODEL_NAME}")
print(f"- Device: {next(model.parameters()).device}")
print(f"- Dtype: {next(model.parameters()).dtype}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B.
401 Client Error. (Request ID: Root=1-69ce4d14-01e4d6be4d96406e63c66ac0;fe9c586c-a29a-41df-877c-e55b1f537909)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
benchmark_results = model_evaluation(
    model_obj=model,
    tokenizer=tokenizer,
    tasks=BENCHMARK_TASKS,
    device=DEVICE,
    limit=BENCHMARK_LIMIT,
    batch_size=BATCH_SIZE,
)

benchmark_rows = []
for task_name, metrics in benchmark_results.items():
    row = {"task": task_name}
    row.update(metrics)
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)
print("\nBenchmark results:")
display(benchmark_df)

In [ ]:
PERFORMANCE_PROMPTS = [
    "Explain what model pruning is in one paragraph.",
    "Summarize why KV cache improves autoregressive generation speed.",
    "List three tradeoffs between model quality and inference efficiency.",
]

memory_perf_runs = []
for prompt in PERFORMANCE_PROMPTS:
    metrics = measure_memory_allocation(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    metrics["prompt"] = prompt
    memory_perf_runs.append(metrics)

memory_perf_df = pd.DataFrame(memory_perf_runs)
performance_summary = {
    "throughput_tokens_s_avg": float(memory_perf_df["throughput_tokens_s"].mean()),
    "throughput_tokens_s_min": float(memory_perf_df["throughput_tokens_s"].min()),
    "throughput_tokens_s_max": float(memory_perf_df["throughput_tokens_s"].max()),
    "static_vram_mb_avg": float(memory_perf_df["static_vram_mb"].mean()),
    "dynamic_delta_mb_avg": float(memory_perf_df["dynamic_delta_mb"].mean()),
}

print("Per-prompt throughput and memory:")
display(memory_perf_df[["prompt", "static_vram_mb", "dynamic_delta_mb", "throughput_tokens_s"]])

print("\nAggregated throughput and memory:")
for k, v in performance_summary.items():
    print(f"- {k}: {v}")

In [ ]:
ENERGY_PROMPTS = [
    "Describe the main difference between pruning and quantization.",
    "Write a short explanation of why batch size affects throughput.",
    "Give two examples of NLP benchmark tasks and what they measure.",
    "Explain in simple words what Joules per token means.",
]

class PromptDataset(Dataset):
    def __init__(self, prompts, tokenizer, max_length=256):
        self.enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

    def __len__(self):
        return self.enc["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
        }

energy_dataset = PromptDataset(ENERGY_PROMPTS, tokenizer)
energy_loader = DataLoader(energy_dataset, batch_size=1, shuffle=False)

In [ ]:
energy_results = measure_energy_consumption(
    model=model,
    tokenizer=tokenizer,
    data_source=energy_loader,
    idle_power_watts=None,
    num_runs=1,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=len(energy_dataset),
)

energy_joules = float(energy_results["energy_net_kwh"] * 3_600_000)
joules_per_token = float(energy_results["efficiency_joules_per_token"])

print("Energy metrics:")
print(f"- energy_net_kwh: {energy_results['energy_net_kwh']}")
print(f"- energy_joules: {energy_joules}")
print(f"- total_tokens: {energy_results['total_tokens']}")
print(f"- duration_sec: {energy_results['duration_sec']}")
print(f"- efficiency_joules_per_token (J/token): {joules_per_token}")
print(f"- co2_emissions_kg: {energy_results['co2_emissions_kg']}")

In [ ]:
final_summary = {
    "config": {
        "MODEL_NAME": MODEL_NAME,
        "DEVICE": DEVICE,
        "BATCH_SIZE": BATCH_SIZE,
        "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        "BENCHMARK_LIMIT": BENCHMARK_LIMIT,
        "BENCHMARK_TASKS": BENCHMARK_TASKS,
    },
    "benchmarks": benchmark_results,
    "performance_memory": {
        "per_prompt": memory_perf_runs,
        "summary": performance_summary,
    },
    "energy": {
        **energy_results,
        "energy_joules": energy_joules,
        "joules_per_token": joules_per_token,
    },
}

print("================ FINAL SUMMARY ================")
print(json.dumps(final_summary, indent=2, default=str))